# RQ2 --- Efficiency

In [1]:
import sys

sys.path.insert(0, "..")
import pandas as pd
from experiments._loader import load_all_results, success_only
from experiments._analysis import setup_matplotlib

setup_matplotlib()
df_all = load_all_results(include_baseline_fail=True)
df = success_only(df_all)
df["total_budget"] = (df["img_budget_used"] + df["txt_budget_used"]).clip(upper=df["budget_max"])
df["budget_utilization"] = df["total_budget"] / df["budget_max"]
df["total_evaluations"] = df["generations_completed"] * 100  # Number of individuals in generation

counts = df["model"].value_counts()
df = df[df["model"].isin(counts[counts >= 15].index)]

In [2]:
METRIC_COLS = [
    "img_budget_used",
    "txt_budget_used",
    "total_budget",
    "total_evaluations",
    "runtime",
]

SCENES = [
    ("MC", "multi"),
    ("SC-MI", "single/multi"),
    ("SC-SI", "single/solo"),
    ("Driving", "udacity"),
]


def format_metric(series: pd.Series) -> str:
    values = series.dropna()

    if values.empty:
        return "---"

    mean = values.mean()
    std = values.std()
    std = 0.0 if pd.isna(std) else std

    return f"${mean:.2f} \\pm {std:.2f}$"


records = []
for model in sorted(df["model"].dropna().unique()):
    for scene_label, obj_category in SCENES:
        scene_mask = (df["model"] == model) & (df["obj_category"] == obj_category)

        for genome_mode in ["multi", "image", "text"]:
            sub = df.loc[scene_mask & (df["genome_mode"] == genome_mode)]

            records.append(
                {
                    "model": model,
                    "scene": scene_label,
                    "genome_mode": genome_mode,
                    "img_budget": format_metric(sub["img_budget_used"]),
                    "txt_budget": format_metric(sub["txt_budget_used"]),
                    "total_budget": format_metric(sub["total_budget"]),
                    "total_evaluations": format_metric(sub["total_evaluations"]),
                    "runtime": format_metric(sub["runtime"]),
                }
            )

efficiency_df = pd.DataFrame(records)

In [ ]:
efficiency_df.to_csv("efficiency.csv")
